# Coni-LCS limits in JAXVacua

**What's in this notebook?** This notebook shows how to work with conifold-LCS (coniLCS) limits in JAXVacua — geometries where one complex structure modulus approaches a conifold singularity while the remaining bulk moduli are at large complex structure.

**In this notebook, you will learn:**
- How to identify conifold directions and perform basis changes with `get_basis_change`
- How to construct models with `limit="coniLCS"` using the logarithmic prepotential near the conifold
- How to compute periods and F-terms in the conifold regime

**Prerequisites:** [Tutorial 12: Moduli space limits](12_moduli_space_limits.ipynb) for the LCS setup, and familiarity with CYTools ([Tutorial 3](../01_basics/3_cytools_interface.ipynb)).

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

In [ ]:
# General imports
import warnings
import numpy as np
from tqdm.auto import tqdm
from scipy.optimize import root

# JAX imports
from jax import jit, vmap
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc

warnings.filterwarnings('ignore')

## Coni-LCS limts in JAXVacua

We study the original example of [2009.03312](https://arxiv.org/pdf/2009.03312) with $h^{1,1}=99$ and $h^{1,2}=3$.
The mirror dual CY can be defined as follows

In [ ]:
from cytools import Polytope
poly = Polytope([[-1,3,-2,-1],[1,-1,0,0],[-1,0,0,1],[-1,0,0,0],[-1,0,1,1],[-1,0,2,0],[-1,0,1,0]])
cy = poly.triangulate().get_cy()
cy

To find the desired conifold curve, let us compute GVs to high degree and check for nilpotent rays:

In [ ]:
gvs = cy.compute_gvs(max_deg=10)
grading_vector = gvs.grading_vec
gvs = gvs.dok
keys = gvs.keys()
curve_charges = np.array(list(keys))
GV_invariants = np.array(list(gvs.values()))

Check for nilpotent curves

In [ ]:
nilpotent_curves = []
for key in keys:
    q = np.array(key)
    degree = q@grading_vector
    if degree==1:
        flag=True
        for i in range(2,20):
            gv=gvs.get(tuple(i*q),0)
            if gv!=0:
                flag=False
                break
        if flag:
            nilpotent_curves.append([q,gvs[key]])
        
nilpotent_curves

These are actually the generators of the toric Mori cone

In [ ]:
mori = cy.toric_mori_cone(in_basis=True)
mori.extremal_rays()

In the example of [2009.03312](https://arxiv.org/pdf/2009.03312), the authors studied the conifold singularity arising when shrinking the curve associated with the class `[-1,1,0]`.
Let us define this as our conifold curve:

In [ ]:
conifold_curve = np.array([-1,1,0])

We can then find a suitable basis transformation in which this curve is represented by the class `[1,0,0]` by calling `jaxvacua.get_basis_change`:

In [ ]:
basis_matrix = jvc.get_basis_change(conifold_curve)
basis_matrix

Let us check that it leads to the expected result

In [ ]:
conifold_curve@basis_matrix.T

This choice is not unique! E.g. the following matrix also works

In [ ]:
basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve@basis_matrix.T

We generate our model as follows

In [ ]:
basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve = np.array([-1,1,0])

model = jvc.FluxVacuaFinder(h12=cy.h11(), 
                                    Q=cy.h11()+cy.h12()+2, 
                                    use_cytools=True, 
                                    mirror_cy=cy, ncf=2,
                                    use_gvs=True,
                                    maximum_degree=6, basis_change=basis_matrix, 
                                    conifold_curve=conifold_curve,
                                    limit="coniLCS",
                                    prange=100,
                                    conifold_basis=True)

We can check that the data obtained matches Eq. (4.42) in [2009.03312](https://arxiv.org/pdf/2009.03312):

In [ ]:
model.lcs_tree.a_matrix,model.lcs_tree.b_vector*24

### Testing examples from main text in [2009.03312](https://arxiv.org/pdf/2009.03312)

Let us reproduce the solution discussed in the main text of section 4.3 of [2009.03312](https://arxiv.org/pdf/2009.03312).
The choices of $M$ and $K$ are

In [ ]:
M = np.array([4,-8,8])
K = np.array([-8,3,-6])

We construct full flux vector as follows

In [ ]:
flux = model.pfv_to_flux(M,K)
flux

Let us use as input for finding the true $F$-term minimum the PFV solution in the two-term racetrack approximation
as stated in [2009.03312](https://arxiv.org/pdf/2009.03312)

In [ ]:
gs = 0.38
tau0 = 1j/gs
z0 = model.pfv_to_moduli(M,K,tau0)
z0

At this initial guess, the $F$-term conditions are approximately satisfied

In [ ]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),flux)

To find the true minimum, we can run the following lines to search for roots of $D_I W$:

In [ ]:
# Get array of real moduli values (for optimisation purposes)
x0 = model._convert_complex_to_real(z0,jnp.conj(z0),tau0,jnp.conj(tau0))

# Run scipy.optimize.root method to find solutions to DW=0
res = root(model.DW_x,x0,args=(flux,),jac=model.dDW_x,tol=1e-10,method="hybr")

res

Let us compute the value of the F-terms

In [ ]:
# Get solution
x1 = res.x

# Compute DW
DW = model.DW_x(x1,flux)

print("DW: ",np.abs(DW))

We check that the solution is inside the facet of the Kähler cone

In [ ]:
z1,_,t1,_ = model._convert_real_to_complex(x1)

flag = np.all(z1.imag@model.lcs_tree.hyperplanes.T>0)&(t1.imag>0)

print("Inside facet of Kähler cone: ",flag)

Compute value of $W_0$ with appropriate normalisation

In [ ]:
W1 = model.superpotential(z1,t1,flux,normalise=True)

print("W0: ",np.abs(W1))

This agree with the expected value $|W_0|\approx 6.9\times 10^{-4}$ stated in Eq. (4.59) of [2009.03312](https://arxiv.org/pdf/2009.03312).

We can also test the value of $z_{cf}$ at the minimum against analytic expectations using the 
linear approximation of the superpotential

In [ ]:
# Compute z_cf analytically (two approximations) and compare to the numerical result.
# `compute_zcf_x` takes the bulk-only real vector (length 2*h12, no z_cf direction);
# `x1[2:]` strips the (Re, Im) of z_cf from the full real vector x1.
zcf_analytic_pfv     = model.compute_zcf_x(x1[2:], flux, mode="pfv")
zcf_analytic_manual = model.compute_zcf_x(x1[2:], flux, mode="autodiff")
zcf_analytic_general = model.compute_zcf_x(x1[2:], flux, mode="manual")
zcf_numerics         = z1[0]

print(f"|z_cf| numerically:              {np.abs(zcf_numerics):.6e}")
print(f"|z_cf| analytically (PFV):       {np.abs(zcf_analytic_pfv):.6e}")
print(f"|z_cf| analytically (bulk VEVs, autodiff): {np.abs(zcf_analytic_general):.6e}")
print(f"|z_cf| analytically (bulk VEVs, manual): {np.abs(zcf_analytic_manual):.6e}")

### Test all minima from [2009.03312](https://arxiv.org/pdf/2009.03312)

We now test the minima quoted in the main text as well as Table 1 in [2009.03312](https://arxiv.org/pdf/2009.03312).
Let us write the PFV data as follows

In [ ]:
Mlist = np.array([[4,-8,8],[4,-8,10],[8,-12,6],[-8,4,12],[-14,6,27]])
Klist = np.array([[-8,3,-6],[-6,3,-4],[-5,1,-2],[5,1,-4],[4,1,-2]])
gslist = np.array([0.38, 0.15, 0.125, 0.35, 0.0643])

Then we find the associated flux vacuum solving $D_IW=0$ as follows:

In [ ]:
results = []

for i in range(len(Mlist)):
    print(f"Testing solution {i+1}/{len(Mlist)} ...")

    M    = Mlist[i]
    K    = Klist[i]
    flux = model.pfv_to_flux(M, K)

    gs   = gslist[i]
    tau0 = 1j / gs
    z0   = model.pfv_to_moduli(M, K, tau0)

    x0  = model._convert_complex_to_real(z0, jnp.conj(z0), tau0, jnp.conj(tau0))
    res = root(model.DW_x, x0, args=(flux,), jac=model.dDW_x, tol=1e-10, method="hybr")
    x1  = res.x

    if not res.success:
        # Hybrid solver stalled — warm-start with a few Newton steps at smaller step size
        z0b, _, tau0b, _ = model._convert_real_to_complex(x1)
        z1n, taun, _     = model.newton_method_flux_vacua(
            z0b, tau0b, flux,
            step_size_Newton=0.1, tol=1e-12, max_iters=100,
            mode="SUSY", solver_mode="real"
        )
        x0  = model._convert_complex_to_real(z1n, jnp.conj(z1n), taun, jnp.conj(taun))
        res = root(model.DW_x, x0, args=(flux,), jac=model.dDW_x, tol=1e-10, method="hybr")
        x1  = res.x

    DW = model.DW_x(x1, flux)
    z1, _, t1, _ = model._convert_real_to_complex(x1)

    in_cone = bool(
        np.all(z1.imag @ model.lcs_tree.hyperplanes.T > 0) and t1.imag > 0
    )
    W0        = float(np.abs(model.superpotential(z1, t1, flux, normalise=True)))
    zcf_num   = complex(z1[0])
    zcf_pfv   = complex(model.compute_zcf_x(x1[2:], flux, mode="pfv"))
    zcf_bulk  = complex(model.compute_zcf_x(x1[2:], flux, mode="manual"))
    dw_norm   = float(np.sum(np.abs(DW)))

    results.append({
        "i":         i,
        "converged": res.success,
        "DW_norm":   dw_norm,
        "in_cone":   in_cone,
        "W0":        W0,
        "zcf_num":   zcf_num,
        "zcf_pfv":   zcf_pfv,
        "zcf_bulk":  zcf_bulk,
    })

print("\nDone.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=150)

sol_labels = [f"Sol. {r['i']+1}" for r in results]
zcf_nums   = [np.abs(r["zcf_num"])  for r in results]
zcf_pfvs   = [np.abs(r["zcf_pfv"])  for r in results]
zcf_bulks  = [np.abs(r["zcf_bulk"]) for r in results]
W0s        = [r["W0"] for r in results]
x_pos      = np.arange(len(results))

# Left panel: compare z_cf estimates
ax = axes[0]
ax.semilogy(x_pos, zcf_nums,  "o-",  label=r"$|z_{\rm cf}|$ numerical",    color="C0", ms=7)
ax.semilogy(x_pos, zcf_pfvs,  "s--", label=r"$|z_{\rm cf}|$ PFV analytic", color="C1", ms=7)
ax.semilogy(x_pos, zcf_bulks, "^:",  label=r"$|z_{\rm cf}|$ bulk VEVs",    color="C2", ms=7)
ax.set_xticks(x_pos)
ax.set_xticklabels(sol_labels, rotation=20)
ax.set_ylabel(r"$|z_{\rm cf}|$", fontsize=12)
ax.set_title(r"Conifold modulus $|z_{\rm cf}|$", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)

# Right panel: |W_0| at each minimum
ax2 = axes[1]
ax2.semilogy(x_pos, W0s, "D-", color="C3", ms=7)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(sol_labels, rotation=20)
ax2.set_ylabel(r"$|W_0|$", fontsize=12)
ax2.set_title(r"Superpotential $|W_0|$ at minimum", fontsize=12)
ax2.grid(True, which="both", alpha=0.3)

plt.suptitle("All solutions from Table 1 of arXiv:2009.03312", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, HTML

header = ["Sol.", "|W₀|", "|z_cf| (num.)", "|z_cf| (PFV)", "|z_cf| (bulk VEVs)", "∑|DW|", "Kähler cone", "Conv."]

rows = []
for r in results:
    rows.append([
        str(r["i"] + 1),
        f"{r['W0']:.3e}",
        f"{np.abs(r['zcf_num']):.3e}",
        f"{np.abs(r['zcf_pfv']):.3e}",
        f"{np.abs(r['zcf_bulk']):.3e}",
        f"{r['DW_norm']:.2e}",
        "✓" if r["in_cone"] else "✗",
        "✓" if r["converged"] else "(warmed)",
    ])

th  = "".join(f"<th style='padding:6px 10px'>{h}</th>" for h in header)
trs = "".join(
    "<tr>" + "".join(f"<td style='padding:5px 10px;text-align:center'>{v}</td>" for v in row) + "</tr>"
    for row in rows
)
display(HTML(f"""
<table border='1' cellspacing='0' style='border-collapse:collapse;font-size:13px'>
  <thead><tr>{th}</tr></thead>
  <tbody>{trs}</tbody>
</table>
"""))

---
## The `coniLCS_series` limit

The `coniLCS_series` limit implements a *linearised* approximation of the conifold prepotential.

Near the conifold locus $z^1 \to 0$, the one-loop worldsheet correction involving the conifold modulus is dominated by the leading logarithm of the dilogarithm.  Concretely, the prepotential takes the form

$$
F_{\rm coniLCS}(z) = F_{\rm poly}(z) + \frac{n_{\rm cf}}{(2\pi i)^3}
    \Bigl[ (2\pi i\, z^1) \ln(2\pi i\, z^1) - (2\pi i\, z^1) \Bigr] + \cdots
$$

The **`coniLCS_series`** mode retains only the *leading linear term* in $z^1$ inside the logarithm, i.e. it replaces the full dilogarithm $\text{Li}_2(e^{2\pi i z^1})$ by its leading-log expansion.  This gives a simpler analytic handle on the vacuum structure when $|z^1| \ll 1$:

$$
F_{\rm coniLCS\_series}(z)
    = F_{\rm poly}(z) + \frac{n_{\rm cf}}{(2\pi i)^2}\, z^1 \ln z^1 + \mathcal{O}(z^1) \,.
$$

**When to use it:**
- As a cheap cross-check: the `coniLCS_series` minimum should match the full `coniLCS` minimum for solutions with $|z_{\rm cf}| \ll 1$.
- For analytic estimates of $z_{\rm cf}$ before running the full Newton solver.
- For solutions with larger $|z_{\rm cf}|$ the series approximation breaks down; use the full `coniLCS` prepotential in that case.

We instantiate a model using `limit="coniLCS_series"` — all other parameters are identical to the `coniLCS` model above:

In [ ]:
model_series = jvc.FluxVacuaFinder(
    h12=cy.h11(), Q=cy.h11()+cy.h12()+2,
    use_cytools=True, 
    mirror_cy=cy, 
    conifold_curve=conifold_curve,
    ncf=2, 
    use_gvs=True,
    maximum_degree=6, 
    basis_change=basis_matrix,
    limit="coniLCS_series",
    prange=10, 
    conifold_basis=True,
    nmax=2
)

We compare `coniLCS` vs `coniLCS_series` for all five solutions. For vacua with $|z_{\rm cf}| \ll 1$ (solutions 1–2) the two prepotentials should give nearly identical $|W_0|$ and $|z_{\rm cf}|$; for the solution with larger $|z_{\rm cf}|$ (solution 4) we expect a visible discrepancy.

In [ ]:
results_series = []

for i in tqdm(range(len(Mlist))):
    M    = Mlist[i]
    K    = Klist[i]
    flux = model_series.pfv_to_flux(M, K)
    gs   = gslist[i]
    tau0 = 1j / gs
    z0   = model_series.pfv_to_moduli(M, K, tau0)

    x0  = model_series._convert_complex_to_real(z0, jnp.conj(z0), tau0, jnp.conj(tau0))
    res = root(model_series.DW_x, x0, args=(flux,), jac=model_series.dDW_x, tol=1e-10, method="hybr")
    x1  = res.x
    
    #print(z0,res)
    
    #sys.exit()

    if not res.success:
        z0b, _, tau0b, _ = model_series._convert_real_to_complex(x1)
        z1n, taun, _     = model_series.newton_method_flux_vacua(
            z0b, tau0b, flux,
            step_size_Newton=0.1, tol=1e-12, max_iters=100,
            mode="SUSY", solver_mode="real"
        )
        x0  = model_series._convert_complex_to_real(z1n, jnp.conj(z1n), taun, jnp.conj(taun))
        res = root(model_series.DW_x, x0, args=(flux,), jac=model_series.dDW_x, tol=1e-10, method="hybr")
        x1  = res.x

    z1, _, t1, _ = model_series._convert_real_to_complex(x1)
    W0_series    = float(np.abs(model_series.superpotential(z1, t1, flux, normalise=True)))
    zcf_series   = float(np.abs(z1[0]))

    results_series.append({"W0": W0_series, "zcf_num": zcf_series, "converged": res.success})

print("Done.")

Let us plot the results below

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=150)

sol_labels   = [f"Sol. {i+1}" for i in range(len(results))]
x_pos        = np.arange(len(results))

W0_full      = [r["W0"] for r in results]
W0_ser       = [r["W0"] for r in results_series]
zcf_full_arr = [np.abs(r["zcf_num"]) for r in results]
zcf_ser_arr  = [r["zcf_num"] for r in results_series]

ax = axes[0]
ax.semilogy(x_pos, W0_full, "o-",  color="C0", label="coniLCS (full)", ms=7)
ax.semilogy(x_pos, W0_ser,  "s--", color="C1", label="coniLCS_series", ms=7)
ax.set_xticks(x_pos)
ax.set_xticklabels(sol_labels, rotation=20)
ax.set_ylabel(r"$|W_0|$", fontsize=12)
ax.set_title(r"$|W_0|$: full vs. series", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)

ax2 = axes[1]
ax2.semilogy(x_pos, zcf_full_arr, "o-",  color="C0", label="coniLCS (full)", ms=7)
ax2.semilogy(x_pos, zcf_ser_arr,  "s--", color="C1", label="coniLCS_series", ms=7)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(sol_labels, rotation=20)
ax2.set_ylabel(r"$|z_{\rm cf}|$", fontsize=12)
ax2.set_title(r"$|z_{\rm cf}|$: full vs. series", fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, which="both", alpha=0.3)

plt.suptitle(
    r"Full coniLCS prepotential vs. linearised series approximation",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated how to work with conifold-LCS limits in JAXVacua:

- **Basis changes** via `get_basis_change` to identify conifold directions
- **Logarithmic prepotential** near the conifold point, including the characteristic $z_{\mathrm{cf}}^2 \ln z_{\mathrm{cf}}$ monodromy term
- **Model construction** with `limit="coniLCS"` for geometries with conifold singularities

For the freezer framework that integrates out the conifold modulus analytically, see [Tutorial 16: Freezer](../06_analysis_and_tools/16_freezer.ipynb). For the theoretical background on conifold PFVs, see the [PFV introduction](../intro/pfv).